# Exploratory Data Analysis - Sales & Inventory Dataset
## Version: XLSX Data Loading

This notebook explores a dataset containing sales, inventory, and product data for 250 products over a 3-year period (2023-2025). The data is designed for demand forecasting, inventory optimization, and supply chain analytics.

**Data source:** `.original_hackathon_data.xlsx` (Excel file with multiple sheets)
- Sheet `Sales` - Monthly sales & inventory (Jan 2023 - Dec 2025, all in one sheet)
- Sheet `Products_parameters` - Static product attributes
- Sheet `Financial_plan` - Annual sales plan/budget targets

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)

## 1. Data Loading (XLSX)

Loading data directly from `.original_hackathon_data.xlsx` which contains all data in a single workbook with sheets: `Sales`, `Products_parameters`, `Financial_plan`.

In [ ]:
# Load all sheets from the original Excel file
xlsx_path = '.original_hackathon_data.xlsx'

xlsx = pd.ExcelFile(xlsx_path)
print(f"Available sheets: {xlsx.sheet_names}")

sales_all = pd.read_excel(xlsx, sheet_name='Sales')
products = pd.read_excel(xlsx, sheet_name='Products_parameters')
financial_plan = pd.read_excel(xlsx, sheet_name='Financial_plan')

# Split sales into train (2023-2024) and test (2025)
id_cols = ['ProductID', 'Product_name', 'Type']
train_cols = [c for c in sales_all.columns if c not in id_cols and str(c).startswith('202') and int(str(c)[:4]) < 2025]
test_cols = [c for c in sales_all.columns if c not in id_cols and str(c).startswith('2025')]

sales_train = sales_all[id_cols + train_cols]
sales_test = sales_all[id_cols + test_cols]

print(f"\nSales All shape: {sales_all.shape}")
print(f"Sales Train shape: {sales_train.shape}")
print(f"Sales Test shape: {sales_test.shape}")
print(f"Products shape: {products.shape}")
print(f"Financial Plan shape: {financial_plan.shape}")

## 2. Data Overview

In [ ]:
print("=" * 60)
print("SALES TRAIN (2023-2024)")
print("=" * 60)
sales_train.head(6)

In [ ]:
print("=" * 60)
print("SALES TEST (2025)")
print("=" * 60)
sales_test.head(6)

In [ ]:
print("=" * 60)
print("PRODUCT PARAMETERS")
print("=" * 60)
products.head(10)

In [ ]:
print("=" * 60)
print("FINANCIAL PLAN")
print("=" * 60)
financial_plan.head(10)

In [ ]:
# Data types and missing values
print("\n--- Sales Train Info ---")
print(sales_train.info())
print(f"\nMissing values: {sales_train.isnull().sum().sum()}")

print("\n--- Products Info ---")
print(products.info())
print(f"\nMissing values: {products.isnull().sum().sum()}")

## 3. Data Transformation - Long Format

The original Excel file stores all months (2023-2025) in a single Sales sheet. Date columns may be read as datetime objects rather than strings. We handle both cases below.

In [ ]:
# Separate Sales and Inventory rows
def transform_to_long(df):
    """Transform wide format sales/inventory data to long format.
    Handles both string and datetime column names (from Excel)."""
    id_cols = ['ProductID', 'Product_name', 'Type']
    value_cols = [c for c in df.columns if c not in id_cols]
    
    long_df = df.melt(id_vars=id_cols, value_vars=value_cols,
                      var_name='Month', value_name='Value')
    
    # Handle both string dates ("2023-01") and datetime objects from Excel
    if long_df['Month'].dtype == 'object':
        long_df['Month'] = pd.to_datetime(long_df['Month'])
    elif not pd.api.types.is_datetime64_any_dtype(long_df['Month']):
        long_df['Month'] = pd.to_datetime(long_df['Month'].astype(str))
    
    return long_df

# Transform train and test
train_long = transform_to_long(sales_train)
test_long = transform_to_long(sales_test)

# Combine all data
all_data_long = pd.concat([train_long, test_long], ignore_index=True)

# Pivot to have Sales and Inventory as separate columns
all_data_pivot = all_data_long.pivot_table(
    index=['ProductID', 'Product_name', 'Month'],
    columns='Type',
    values='Value'
).reset_index()
all_data_pivot.columns.name = None

print(f"Combined long format shape: {all_data_pivot.shape}")
all_data_pivot.head(10)

## 4. Sales Analysis

In [ ]:
# Total sales per product (all years)
total_sales = all_data_pivot.groupby(['ProductID', 'Product_name'])['Sales'].sum().reset_index()
total_sales = total_sales.sort_values('Sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 15 products by total sales
axes[0].barh(total_sales.head(15)['Product_name'], total_sales.head(15)['Sales'], color='steelblue')
axes[0].set_title('Top 15 Products by Total Sales (Units)', fontsize=13)
axes[0].set_xlabel('Total Units Sold')
axes[0].invert_yaxis()

# Bottom 15 products by total sales
axes[1].barh(total_sales.tail(15)['Product_name'], total_sales.tail(15)['Sales'], color='coral')
axes[1].set_title('Bottom 15 Products by Total Sales (Units)', fontsize=13)
axes[1].set_xlabel('Total Units Sold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Monthly total sales across all products
monthly_total = all_data_pivot.groupby('Month')['Sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_total['Month'], monthly_total['Sales'], marker='o', linewidth=2, color='steelblue')
ax.axvline(pd.Timestamp('2025-01-01'), color='red', linestyle='--', alpha=0.7, label='Train/Test Split')
ax.set_title('Total Monthly Sales Across All Products', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Total Units Sold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Sales distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_data_pivot['Sales'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Monthly Sales (All Products)', fontsize=13)
axes[0].set_xlabel('Units Sold')
axes[0].set_ylabel('Frequency')

# Log scale
axes[1].hist(all_data_pivot['Sales'].clip(lower=1), bins=50, color='teal', edgecolor='white', alpha=0.8)
axes[1].set_xscale('log')
axes[1].set_title('Distribution of Monthly Sales (Log Scale)', fontsize=13)
axes[1].set_xlabel('Units Sold (log)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f"\nSales Statistics:")
print(all_data_pivot['Sales'].describe())

## 5. Inventory Analysis

In [ ]:
# Stockout analysis (Inventory_level = 0)
stockouts = all_data_pivot[all_data_pivot['Inventory_level'] == 0]
stockout_counts = stockouts.groupby('ProductID').size().reset_index(name='Stockout_Months')
stockout_counts = stockout_counts.sort_values('Stockout_Months', ascending=False)

print(f"Total stockout events: {len(stockouts)}")
print(f"Products with at least one stockout: {len(stockout_counts)}")
print(f"\nTop 10 products by stockout frequency:")
print(stockout_counts.head(10).to_string(index=False))

In [ ]:
# Inventory level trends for sample products
sample_products = ['P001', 'P005', 'P008', 'P015']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, pid in enumerate(sample_products):
    product_data = all_data_pivot[all_data_pivot['ProductID'] == pid]
    ax = axes[i]
    ax.plot(product_data['Month'], product_data['Sales'], label='Sales', color='steelblue', linewidth=1.5)
    ax.plot(product_data['Month'], product_data['Inventory_level'], label='Inventory', color='orange', linewidth=1.5)
    ax.axvline(pd.Timestamp('2025-01-01'), color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{pid} - Sales vs Inventory', fontsize=12)
    ax.legend()
    ax.set_xlabel('Month')

plt.tight_layout()
plt.show()

## 6. Product Parameters Analysis

In [ ]:
# Product parameters summary statistics
print("Product Parameters Summary:")
print(products.describe().round(2))

In [ ]:
# Distributions of product parameters
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
params = ['Storage_cost_PLN_per_unit_month', 'Shelf_life_months', 'Lead_time_months',
           'Safety_stock_months', 'Price_PLN_per_unit']
colors = ['steelblue', 'coral', 'teal', 'orange', 'purple']

for i, (param, color) in enumerate(zip(params, colors)):
    ax = axes.flatten()[i]
    if param == 'Lead_time_months':
        ax.hist(products[param], bins=range(int(products[param].min()), int(products[param].max()) + 2), color=color, edgecolor='white', alpha=0.8, align='left')
        ax.xaxis.set_major_locator(plt.MultipleLocator(1))
    else:
        ax.hist(products[param], bins=20, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(param.replace('_', ' '), fontsize=11)
    ax.set_xlabel(param.split('_')[-1] if 'PLN' not in param else 'PLN')

# Correlation heatmap in last subplot
ax = axes.flatten()[5]
corr = products.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Parameter Correlations', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Price vs Storage cost
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(products['Price_PLN_per_unit'], 
                     products['Storage_cost_PLN_per_unit_month'],
                     c=products['Shelf_life_months'], 
                     cmap='viridis', s=60, alpha=0.7, edgecolors='gray')
plt.colorbar(scatter, label='Shelf Life (months)')
ax.set_xlabel('Price (PLN per unit)')
ax.set_ylabel('Storage Cost (PLN per unit/month)')
ax.set_title('Price vs Storage Cost (colored by Shelf Life)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Financial Plan Analysis

In [ ]:
# Plan vs Actual comparison (2023 & 2024)
# Calculate actual annual sales from training data
sales_only = all_data_pivot[['ProductID', 'Month', 'Sales']].copy()
sales_only['Year'] = sales_only['Month'].dt.year

actual_annual = sales_only.groupby(['ProductID', 'Year'])['Sales'].sum().reset_index()
actual_annual = actual_annual.rename(columns={'Sales': 'Actual_Units'})

# Merge with plan
plan_melted = financial_plan.melt(
    id_vars=['ProductID', 'Product_name', 'Price_PLN_per_unit'],
    value_vars=['Plan_2023_units', 'Plan_2024_units', 'Plan_2025_units'],
    var_name='Plan_Year', value_name='Plan_Units'
)
plan_melted['Year'] = plan_melted['Plan_Year'].str.extract(r'(\d{4})').astype(int)

comparison = actual_annual.merge(plan_melted[['ProductID', 'Year', 'Plan_Units']], on=['ProductID', 'Year'])
comparison['Plan_Achievement_pct'] = (comparison['Actual_Units'] / comparison['Plan_Units'] * 100).round(1)

print("Plan Achievement Summary (%) by Year:")
print(comparison.groupby('Year')['Plan_Achievement_pct'].describe().round(1))

In [ ]:
# Plan achievement distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, year in enumerate([2023, 2024, 2025]):
    year_data = comparison[comparison['Year'] == year]
    if len(year_data) > 0:
        axes[i].hist(year_data['Plan_Achievement_pct'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
        axes[i].axvline(100, color='red', linestyle='--', label='100% Target')
        axes[i].set_title(f'Plan Achievement {year} (%)', fontsize=12)
        axes[i].set_xlabel('Achievement %')
        axes[i].legend()
    else:
        axes[i].set_title(f'Plan Achievement {year} - No Data', fontsize=12)

plt.tight_layout()
plt.show()

## 8. Seasonality Analysis

In [ ]:
# Monthly seasonality patterns
all_data_pivot['Month_num'] = all_data_pivot['Month'].dt.month
all_data_pivot['Year'] = all_data_pivot['Month'].dt.year

monthly_avg = all_data_pivot.groupby('Month_num')['Sales'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall seasonality
axes[0].bar(monthly_avg['Month_num'], monthly_avg['Sales'], color='steelblue', edgecolor='white')
axes[0].set_title('Average Sales by Month (All Products)', fontsize=13)
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Units Sold')
axes[0].set_xticks(range(1, 13))

# Year-over-year comparison
yearly_monthly = all_data_pivot.groupby(['Year', 'Month_num'])['Sales'].sum().reset_index()
for year in [2023, 2024, 2025]:
    year_data = yearly_monthly[yearly_monthly['Year'] == year]
    axes[1].plot(year_data['Month_num'], year_data['Sales'], marker='o', label=str(year), linewidth=2)

axes[1].set_title('Total Monthly Sales by Year', fontsize=13)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Total Units Sold')
axes[1].set_xticks(range(1, 13))
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Key Metrics & Summary

In [ ]:
# Merge product parameters with sales summary
product_summary = total_sales.merge(products, on=['ProductID', 'Product_name'])
product_summary['Total_Revenue_PLN'] = product_summary['Sales'] * product_summary['Price_PLN_per_unit']
product_summary['Total_Storage_Cost_PLN'] = (
    product_summary['Storage_cost_PLN_per_unit_month'] * 
    all_data_pivot.groupby('ProductID')['Inventory_level'].mean().values * 36  # 36 months
)

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total products: {len(products)}")
print(f"Time period: Jan 2023 - Dec 2025 (36 months)")
print(f"Training period: 24 months | Test period: 12 months")
print(f"\nTotal units sold (all time): {all_data_pivot['Sales'].sum():,.0f}")
print(f"Average monthly sales per product: {all_data_pivot['Sales'].mean():.1f} units")
print(f"\nStockout events (inventory=0): {(all_data_pivot['Inventory_level'] == 0).sum()}")
print(f"Products experiencing stockouts: {len(stockout_counts)}")
print(f"\nPrice range: {products['Price_PLN_per_unit'].min():.2f} - {products['Price_PLN_per_unit'].max():.2f} PLN")
print(f"Lead time range: {products['Lead_time_months'].min()} - {products['Lead_time_months'].max()} months")
print(f"Shelf life range: {products['Shelf_life_months'].min()} - {products['Shelf_life_months'].max()} months")

In [ ]:
# Top products by estimated revenue
top_revenue = product_summary.nlargest(15, 'Total_Revenue_PLN')[['ProductID', 'Product_name', 'Sales', 'Price_PLN_per_unit', 'Total_Revenue_PLN']]
top_revenue['Total_Revenue_PLN'] = top_revenue['Total_Revenue_PLN'].round(2)

print("\nTop 15 Products by Estimated Revenue (PLN):")
print(top_revenue.to_string(index=False))